# Day 1 data cleaning and analytical modeling

This notebook reads only from `data/raw`, creates reproducible order-level and native-grain processed tables, and validates the requested model. It does not perform business analysis or modify raw files.

In [1]:
from pathlib import Path
import json
import pandas as pd

RAW = Path('data/raw')
PROCESSED = Path('data/processed')
PROCESSED.mkdir(parents=True, exist_ok=True)

orders = pd.read_csv(RAW / 'olist_orders_dataset.csv', parse_dates=[
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
])
customers = pd.read_csv(RAW / 'olist_customers_dataset.csv', dtype={'customer_zip_code_prefix': 'Int64'})
items = pd.read_csv(RAW / 'olist_order_items_dataset.csv', parse_dates=['shipping_limit_date'])
payments = pd.read_csv(RAW / 'olist_order_payments_dataset.csv')
reviews = pd.read_csv(RAW / 'olist_order_reviews_dataset.csv')
products = pd.read_csv(RAW / 'olist_products_dataset.csv')
sellers = pd.read_csv(RAW / 'olist_sellers_dataset.csv', dtype={'seller_zip_code_prefix': 'Int64'})
translations = pd.read_csv(RAW / 'product_category_name_translation.csv')

for frame_name, frame in {'orders': orders, 'customers': customers, 'items': items, 'payments': payments, 'reviews': reviews, 'products': products, 'sellers': sellers, 'translations': translations}.items():
    print(f'{frame_name}: {len(frame):,} rows')

orders: 99,441 rows
customers: 99,441 rows
items: 112,650 rows
payments: 103,886 rows
reviews: 99,224 rows
products: 32,951 rows
sellers: 3,095 rows
translations: 71 rows


In [2]:
# Dimension tables. Product physical attributes remain missing when missing in the source.
dim_customers = customers.rename(columns={
    'customer_zip_code_prefix': 'customer_zip_prefix',
})[['customer_id', 'customer_unique_id', 'customer_city', 'customer_state', 'customer_zip_prefix']].copy()

product_lookup = products.merge(translations, on='product_category_name', how='left', validate='many_to_one')
product_lookup['product_category_name_english'] = product_lookup['product_category_name_english'].fillna(product_lookup['product_category_name'])
product_lookup['product_category_name_english'] = product_lookup['product_category_name_english'].fillna('Unknown')
dim_products = product_lookup.rename(columns={
    'product_category_name': 'product_category_original',
    'product_category_name_english': 'product_category_english',
})[['product_id', 'product_category_original', 'product_category_english',
    'product_name_lenght', 'product_description_lenght', 'product_photos_qty',
    'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']].copy()

dim_sellers = sellers.rename(columns={'seller_zip_code_prefix': 'seller_zip_prefix'})[[
    'seller_id', 'seller_city', 'seller_state', 'seller_zip_prefix'
]].copy()

# Native-grain order-item fact. The category display fallback is applied after the product join.
fact_order_items = items.merge(product_lookup[['product_id', 'product_category_name', 'product_category_name_english']],
                               on='product_id', how='left', validate='many_to_one')
fact_order_items = fact_order_items.merge(orders[['order_id', 'customer_id']], on='order_id', how='left', validate='many_to_one')
fact_order_items = fact_order_items.merge(customers[['customer_id', 'customer_state']], on='customer_id', how='left', validate='many_to_one')
fact_order_items = fact_order_items.merge(sellers[['seller_id', 'seller_state']], on='seller_id', how='left', validate='many_to_one')
fact_order_items['product_category_name_english'] = fact_order_items['product_category_name_english'].fillna(fact_order_items['product_category_name'])
fact_order_items['product_category_name_english'] = fact_order_items['product_category_name_english'].fillna('Unknown')
fact_order_items = fact_order_items.rename(columns={
    'product_category_name': 'product_category_original',
    'product_category_name_english': 'product_category_english',
    'shipping_limit_date': 'shipping_limit_timestamp',
})[['order_id', 'order_item_id', 'product_id', 'seller_id', 'product_category_original',
    'product_category_english', 'shipping_limit_timestamp', 'price', 'freight_value',
    'customer_state', 'seller_state']].copy()

# Aggregate every one-to-many source before joining to the order grain.
item_agg = fact_order_items.groupby('order_id', as_index=False).agg(
    item_count=('order_item_id', 'size'),
    seller_count=('seller_id', 'nunique'),
    category_count=('product_category_english', 'nunique'),
    merchandise_value=('price', 'sum'),
    freight_value=('freight_value', 'sum'),
)
payment_agg = payments.groupby('order_id', as_index=False).agg(
    payment_record_count=('payment_sequential', 'size'),
    payment_value=('payment_value', 'sum'),
)
review_agg = reviews.groupby('order_id', as_index=False).agg(
    review_count=('review_score', 'size'),
    average_review_score=('review_score', 'mean'),
)

fact_orders = orders.rename(columns={
    'order_purchase_timestamp': 'purchase_timestamp',
    'order_approved_at': 'approval_timestamp',
    'order_delivered_carrier_date': 'carrier_handoff_timestamp',
    'order_delivered_customer_date': 'delivery_timestamp',
    'order_estimated_delivery_date': 'estimated_delivery_date',
}).merge(dim_customers, on='customer_id', how='left', validate='many_to_one')
fact_orders = fact_orders.merge(item_agg, on='order_id', how='left', validate='one_to_one')
fact_orders = fact_orders.merge(payment_agg, on='order_id', how='left', validate='one_to_one')
fact_orders = fact_orders.merge(review_agg, on='order_id', how='left', validate='one_to_one')

# Date-derived fields use calendar dates for purchase month and delivery classification.
fact_orders['purchase_date'] = fact_orders['purchase_timestamp'].dt.date
fact_orders['purchase_year_month'] = fact_orders['purchase_timestamp'].dt.to_period('M').astype('string')

def valid_sequence(start, end):
    return pd.Series(pd.NA, index=fact_orders.index, dtype='boolean').where(start.isna() | end.isna(), end >= start)

fact_orders['valid_approval_sequence'] = valid_sequence(fact_orders['purchase_timestamp'], fact_orders['approval_timestamp']).astype('boolean')
fact_orders['valid_preparation_sequence'] = valid_sequence(fact_orders['approval_timestamp'], fact_orders['carrier_handoff_timestamp']).astype('boolean')
fact_orders['valid_transportation_sequence'] = valid_sequence(fact_orders['carrier_handoff_timestamp'], fact_orders['delivery_timestamp']).astype('boolean')
fact_orders['valid_total_lead_time_sequence'] = valid_sequence(fact_orders['purchase_timestamp'], fact_orders['delivery_timestamp'])

fact_orders['approval_hours'] = (fact_orders['approval_timestamp'] - fact_orders['purchase_timestamp']).dt.total_seconds().div(3600).where(fact_orders['valid_approval_sequence'].eq(True))
fact_orders['preparation_days'] = (fact_orders['carrier_handoff_timestamp'] - fact_orders['approval_timestamp']).dt.total_seconds().div(86400).where(fact_orders['valid_preparation_sequence'].eq(True))
fact_orders['transportation_days'] = (fact_orders['delivery_timestamp'] - fact_orders['carrier_handoff_timestamp']).dt.total_seconds().div(86400).where(fact_orders['valid_transportation_sequence'].eq(True))
fact_orders['total_lead_time_days'] = (fact_orders['delivery_timestamp'] - fact_orders['purchase_timestamp']).dt.total_seconds().div(86400).where(fact_orders['valid_total_lead_time_sequence'].eq(True))

delivered_with_dates = fact_orders['order_status'].eq('delivered') & fact_orders['delivery_timestamp'].notna()
fact_orders['delivery_variance_days'] = (fact_orders['delivery_timestamp'].dt.normalize() - fact_orders['estimated_delivery_date'].dt.normalize()).dt.days.where(delivered_with_dates)
fact_orders['delivery_classification'] = 'Not Classified'
fact_orders.loc[delivered_with_dates & fact_orders['delivery_variance_days'].le(0), 'delivery_classification'] = 'On Time'
fact_orders.loc[delivered_with_dates & fact_orders['delivery_variance_days'].gt(0), 'delivery_classification'] = 'Late'
fact_orders['delay_days'] = fact_orders['delivery_variance_days'].where(fact_orders['delivery_variance_days'].gt(0))

for col in ['item_count', 'seller_count', 'category_count', 'payment_record_count', 'review_count']:
    fact_orders[col] = fact_orders[col].fillna(0).astype('int64')
fact_orders['has_item_data'] = fact_orders['item_count'].gt(0)
fact_orders['has_review_data'] = fact_orders['review_count'].gt(0)

order_columns = ['order_id', 'customer_id', 'customer_unique_id', 'customer_city', 'customer_state',
    'order_status', 'purchase_timestamp', 'purchase_date', 'purchase_year_month', 'approval_timestamp',
    'carrier_handoff_timestamp', 'delivery_timestamp', 'estimated_delivery_date', 'approval_hours',
    'preparation_days', 'transportation_days', 'total_lead_time_days', 'delivery_variance_days', 'delay_days',
    'delivery_classification', 'valid_approval_sequence', 'valid_preparation_sequence',
    'valid_transportation_sequence', 'item_count', 'seller_count', 'category_count', 'merchandise_value',
    'freight_value', 'payment_record_count', 'payment_value', 'review_count', 'average_review_score',
    'has_item_data', 'has_review_data']
fact_orders = fact_orders[order_columns]

# Write only processed outputs. Dates are serialized in reproducible ISO-like form by pandas.
outputs = {
    'fact_orders.csv': fact_orders,
    'fact_order_items.csv': fact_order_items,
    'dim_customers.csv': dim_customers,
    'dim_products.csv': dim_products,
    'dim_sellers.csv': dim_sellers,
}
for filename, frame in outputs.items():
    if frame.columns.duplicated().any():
        raise AssertionError(f'Duplicate columns in {filename}')
    frame.to_csv(PROCESSED / filename, index=False)
    print(f'wrote {filename}: {len(frame):,} rows')

wrote fact_orders.csv: 99,441 rows
wrote fact_order_items.csv: 112,650 rows
wrote dim_customers.csv: 99,441 rows
wrote dim_products.csv: 32,951 rows
wrote dim_sellers.csv: 3,095 rows


In [3]:
# Independent validation checks. No analytical results or recommendations are produced here.
ABS_TOL = 1e-6
REL_TOL = 0.0

def within_tolerance(left, right):
    return abs(float(left) - float(right)) <= ABS_TOL + REL_TOL * abs(float(right))

def check(name, condition):
    value = bool(condition)
    validations[name] = {'passed': value}
    if not value:
        raise AssertionError(name)

validations = {}
check('fact_orders row count equals raw orders', len(fact_orders) == len(orders))
check('fact_orders order_id unique', fact_orders['order_id'].is_unique)
check('fact_order_items row count equals raw order-items', len(fact_order_items) == len(items))
check('fact_order_items order_id + order_item_id unique', not fact_order_items.duplicated(['order_id', 'order_item_id']).any())
check('dim_customers key unique', dim_customers['customer_id'].is_unique)
check('dim_products key unique', dim_products['product_id'].is_unique)
check('dim_sellers key unique', dim_sellers['seller_id'].is_unique)
check('no order count inflation after aggregate joins', len(fact_orders) == len(orders))
item_price_difference = float(fact_order_items['price'].sum() - items['price'].sum())
item_freight_difference = float(fact_order_items['freight_value'].sum() - items['freight_value'].sum())
order_merchandise_difference = float(fact_orders['merchandise_value'].sum() - items['price'].sum())
order_freight_difference = float(fact_orders['freight_value'].sum() - items['freight_value'].sum())
payment_difference = float(fact_orders['payment_value'].sum() - payments['payment_value'].sum())
check('item price total preserved within tolerance', within_tolerance(fact_order_items['price'].sum(), items['price'].sum()))
check('item freight total preserved within tolerance', within_tolerance(fact_order_items['freight_value'].sum(), items['freight_value'].sum()))
check('order merchandise total reconciles within tolerance', within_tolerance(fact_orders['merchandise_value'].sum(), items['price'].sum()))
check('order freight total reconciles within tolerance', within_tolerance(fact_orders['freight_value'].sum(), items['freight_value'].sum()))
check('payment totals reconcile within tolerance', within_tolerance(fact_orders['payment_value'].sum(), payments['payment_value'].sum()))
check('classification values allowed', set(fact_orders['delivery_classification'].dropna().unique()) <= {'On Time', 'Late', 'Not Classified'})
check('same-date deliveries are On Time', fact_orders.loc[fact_orders['delivery_timestamp'].notna() & fact_orders['estimated_delivery_date'].notna() & fact_orders['delivery_timestamp'].dt.date.eq(fact_orders['estimated_delivery_date'].dt.date) & fact_orders['order_status'].eq('delivered'), 'delivery_classification'].eq('On Time').all())
check('invalid approval sequences have null duration', fact_orders.loc[fact_orders['valid_approval_sequence'].eq(False), 'approval_hours'].isna().all())
check('invalid preparation sequences have null duration', fact_orders.loc[fact_orders['valid_preparation_sequence'].eq(False), 'preparation_days'].isna().all())
check('invalid transportation sequences have null duration', fact_orders.loc[fact_orders['valid_transportation_sequence'].eq(False), 'transportation_days'].isna().all())
check('processed outputs have no duplicate columns', all(not frame.columns.duplicated().any() for frame in outputs.values()))

validation_results = {
    'row_counts': {name: len(frame) for name, frame in outputs.items()},
    'raw_row_counts': {'orders': len(orders), 'order_items': len(items), 'payments': len(payments), 'reviews': len(reviews)},
    'invalid_sequence_counts': {
        'approval': int(fact_orders['valid_approval_sequence'].eq(False).sum()),
        'preparation': int(fact_orders['valid_preparation_sequence'].eq(False).sum()),
        'transportation': int(fact_orders['valid_transportation_sequence'].eq(False).sum()),
    },
    'same_date_on_time_count': int((fact_orders['delivery_classification'].eq('On Time') & fact_orders['delivery_variance_days'].eq(0)).sum()),
    'monetary_tolerance': {'absolute': ABS_TOL, 'relative': REL_TOL},
    'monetary_differences': {
        'item_price': item_price_difference,
        'item_freight': item_freight_difference,
        'order_merchandise': order_merchandise_difference,
        'order_freight': order_freight_difference,
        'payment': payment_difference,
    },
    'checks': validations,
}
print(json.dumps(validation_results, indent=2, default=str))

{
  "row_counts": {
    "fact_orders.csv": 99441,
    "fact_order_items.csv": 112650,
    "dim_customers.csv": 99441,
    "dim_products.csv": 32951,
    "dim_sellers.csv": 3095
  },
  "raw_row_counts": {
    "orders": 99441,
    "order_items": 112650,
    "payments": 103886,
    "reviews": 99224
  },
  "invalid_sequence_counts": {
    "approval": 0,
    "preparation": 1359,
    "transportation": 23
  },
  "same_date_on_time_count": 1292,
  "monetary_tolerance": {
    "absolute": 1e-06,
    "relative": 0.0
  },
  "monetary_differences": {
    "item_price": 0.0,
    "item_freight": 0.0,
    "order_merchandise": 0.0,
    "order_freight": 0.0,
    "payment": 0.0
  },
  "checks": {
    "fact_orders row count equals raw orders": {
      "passed": true
    },
    "fact_orders order_id unique": {
      "passed": true
    },
    "fact_order_items row count equals raw order-items": {
      "passed": true
    },
    "fact_order_items order_id + order_item_id unique": {
      "passed": true
    },